In [1]:
# 检查环境，暂时不安装或更换依赖
import sys
import torch
import numpy as np

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cpu
NumPy version: 2.1.3
GPU available: False


In [2]:
# 上传 MATLAB 生成的压缩包
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]

assert len(zip_names) == 1, "Please upload one ZIP file."

# 解压到固定实验目录
pilot_dir = Path("/content/ELEC5305_Pilot")
pilot_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_names[0], "r") as archive:
    archive.extractall(pilot_dir)

# 检查音频和参考文本
audio_dir = pilot_dir / "results" / "audio"
audio_files = list(audio_dir.glob("*.wav"))
manifest_path = pilot_dir / "utterances.csv"

assert len(audio_files) == 40, "Expected 40 WAV files."
assert manifest_path.is_file(), "Reference transcript file not found."

print("Audio files:", len(audio_files))
print("Transcript file:", manifest_path)
print("Upload and extraction completed.")

Saving pilot_upload.zip to pilot_upload.zip
Audio files: 40
Transcript file: /content/ELEC5305_Pilot/utterances.csv
Upload and extraction completed.


In [3]:
# 安装语音识别和评价工具
%pip install -q openai-whisper jiwer soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.7 MB/s eta 0:00:00


In [4]:
# 检查安装结果和运行环境
import sys
import torch
import whisper
import jiwer
import soundfile as sf
from importlib.metadata import version

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Whisper version:", version("openai-whisper"))
print("JiWER version:", version("jiwer"))
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

print("Installation check completed.")

Python version: 3.13.16
PyTorch version: 2.11.0+cpu
Whisper version: 20250625
JiWER version: 4.0.0
GPU available: False
Installation check completed.


In [5]:
# 加载模型，不进行训练
import torch
import whisper

device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("base.en", device=device)
model.eval()

# 后续所有音频都使用相同设置
decode_settings = {
    "language": "en",
    "task": "transcribe",
    "temperature": 0.0,
    "beam_size": 5,
    "fp16": False,
    "condition_on_previous_text": False,
    "compression_ratio_threshold": 2.4,
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "verbose": None,
}

print("Model: base.en")
print("Device:", device)
print("Model loaded successfully.")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 113MiB/s]


Model: base.en
Device: cpu
Model loaded successfully.


In [6]:
# 读取文件清单
import csv
import json
import soundfile as sf
import jiwer
from pathlib import Path
from whisper.normalizers import EnglishTextNormalizer
from importlib.metadata import version

pilot_dir = Path("/content/ELEC5305_Pilot")
audio_dir = pilot_dir / "results" / "audio"
output_dir = pilot_dir / "results" / "asr"
output_dir.mkdir(parents=True, exist_ok=True)

with (pilot_dir / "utterances.csv").open(
    encoding="utf-8-sig", newline=""
) as file:
    manifest = list(csv.DictReader(file))

assert len(manifest) == 10, "Expected 10 utterances."

normalizer = EnglishTextNormalizer()
clean_results = []

# 逐条识别，保留原始文本和规范化文本
for index, row in enumerate(manifest, start=1):
    utterance_id = row["utterance_id"]
    audio_path = audio_dir / f"{utterance_id}_clean.wav"

    audio, sample_rate = sf.read(audio_path, dtype="float32")
    assert sample_rate == 16000, "Expected 16 kHz audio."
    assert audio.ndim == 1, "Expected mono audio."

    with torch.inference_mode():
        result = model.transcribe(audio, **decode_settings)

    reference = row["reference_text"]
    hypothesis = result["text"].strip()

    clean_results.append({
        "utterance_id": utterance_id,
        "condition": "clean",
        "reference_text": reference,
        "hypothesis_text": hypothesis,
        "reference_normalized": normalizer(reference),
        "hypothesis_normalized": normalizer(hypothesis),
    })

    print(f"[{index}/10] {utterance_id}")
    print("Reference:", reference)
    print("Whisper:", hypothesis)
    print()

# 计算整体 WER，而不是平均每句话的 WER
references = [row["reference_normalized"] for row in clean_results]
hypotheses = [row["hypothesis_normalized"] for row in clean_results]

assert all(references), "An empty reference was found."
scores = jiwer.process_words(references, hypotheses)

print(f"Clean WER: {scores.wer * 100:.2f}%")
print("Substitutions:", scores.substitutions)
print("Deletions:", scores.deletions)
print("Insertions:", scores.insertions)

# 保存识别文本
with (output_dir / "clean_transcripts.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=list(clean_results[0]))
    writer.writeheader()
    writer.writerows(clean_results)

# 保存模型设置和软件版本
settings = {
    "model": "base.en",
    "device": device,
    "decode_settings": decode_settings,
    "normalizer": "Whisper EnglishTextNormalizer",
    "whisper_version": version("openai-whisper"),
    "jiwer_version": version("jiwer"),
    "torch_version": torch.__version__,
}

with (output_dir / "recognition_settings.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(settings, file, indent=2)

print("Clean recognition results saved.")

[1/10] 1272-135031-0000
Reference: BECAUSE YOU WERE SLEEPING INSTEAD OF CONQUERING THE LOVELY ROSE PRINCESS HAS BECOME A FIDDLE WITHOUT A BOW WHILE POOR SHAGGY SITS THERE A COOING DOVE
Whisper: Because you are sleeping instead of conquering, the lovely rose princess has become a fiddle without a bow, while Poroshenko sits there, accuing dove.

[2/10] 1462-170142-0000
Reference: THE LAST TWO DAYS OF THE VOYAGE BARTLEY FOUND ALMOST INTOLERABLE
Whisper: The last two days of the voyage, Bartley found almost intolerable.

[3/10] 174-168635-0000
Reference: HE HAD NEVER BEEN FATHER LOVER HUSBAND FRIEND
Whisper: He had never been father, lover, husband, friend.

[4/10] 1988-147956-0000
Reference: FUCHS BROUGHT UP A SACK OF POTATOES AND A PIECE OF CURED PORK FROM THE CELLAR AND GRANDMOTHER PACKED SOME LOAVES OF SATURDAY'S BREAD A JAR OF BUTTER AND SEVERAL PUMPKIN PIES IN THE STRAW OF THE WAGON BOX
Whisper: Folks brought up a sack of potatoes and a piece of cured pork from the cellar. And grandm

In [7]:
# 识别带噪语音，保持模型和设置不变
noisy_results = []

for index, row in enumerate(manifest, start=1):
    utterance_id = row["utterance_id"]
    audio_path = audio_dir / f"{utterance_id}_noisy.wav"

    audio, sample_rate = sf.read(audio_path, dtype="float32")
    assert sample_rate == 16000, "Expected 16 kHz audio."
    assert audio.ndim == 1, "Expected mono audio."

    with torch.inference_mode():
        result = model.transcribe(audio, **decode_settings)

    reference = row["reference_text"]
    hypothesis = result["text"].strip()

    noisy_results.append({
        "utterance_id": utterance_id,
        "condition": "noisy",
        "reference_text": reference,
        "hypothesis_text": hypothesis,
        "reference_normalized": normalizer(reference),
        "hypothesis_normalized": normalizer(hypothesis),
    })

    print(f"[{index}/10] {utterance_id}")
    print("Reference:", reference)
    print("Whisper:", hypothesis)
    print()

# 计算整体 WER
noisy_scores = jiwer.process_words(
    [row["reference_normalized"] for row in noisy_results],
    [row["hypothesis_normalized"] for row in noisy_results],
)

print(f"Noisy WER: {noisy_scores.wer * 100:.2f}%")
print("Substitutions:", noisy_scores.substitutions)
print("Deletions:", noisy_scores.deletions)
print("Insertions:", noisy_scores.insertions)

# 保存识别文本
with (output_dir / "noisy_transcripts.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=list(noisy_results[0]))
    writer.writeheader()
    writer.writerows(noisy_results)

# 从已保存的干净语音结果重新计算基准
clean_scores = jiwer.process_words(
    [row["reference_normalized"] for row in clean_results],
    [row["hypothesis_normalized"] for row in clean_results],
)

print()
print(f"Clean WER: {clean_scores.wer * 100:.2f}%")
print(f"Noisy WER: {noisy_scores.wer * 100:.2f}%")
print("Noisy recognition results saved.")

[1/10] 1272-135031-0000
Reference: BECAUSE YOU WERE SLEEPING INSTEAD OF CONQUERING THE LOVELY ROSE PRINCESS HAS BECOME A FIDDLE WITHOUT A BOW WHILE POOR SHAGGY SITS THERE A COOING DOVE
Whisper: Because you are sleeping instead of conquering, the lovely rose princess has become a fiddle without a bow, while poor Shaggy sits there accuing dove.

[2/10] 1462-170142-0000
Reference: THE LAST TWO DAYS OF THE VOYAGE BARTLEY FOUND ALMOST INTOLERABLE
Whisper: The last two days of the voyage, Bartley found almost intolerable.

[3/10] 174-168635-0000
Reference: HE HAD NEVER BEEN FATHER LOVER HUSBAND FRIEND
Whisper: He had never been father, lover, husband, friend.

[4/10] 1988-147956-0000
Reference: FUCHS BROUGHT UP A SACK OF POTATOES AND A PIECE OF CURED PORK FROM THE CELLAR AND GRANDMOTHER PACKED SOME LOAVES OF SATURDAY'S BREAD A JAR OF BUTTER AND SEVERAL PUMPKIN PIES IN THE STRAW OF THE WAGON BOX
Whisper: Folks brought up a sack of potatoes and a piece of cured pork from the cellar. And grandm

In [8]:
# 使用同一个模型识别两种经典降噪后的音频
classical_scores = {}

for condition in ["spectral_subtraction", "wiener"]:
    condition_results = []
    print("Processing condition:", condition)

    for index, row in enumerate(manifest, start=1):
        utterance_id = row["utterance_id"]
        audio_path = audio_dir / f"{utterance_id}_{condition}.wav"

        audio, sample_rate = sf.read(audio_path, dtype="float32")
        assert sample_rate == 16000, "Expected 16 kHz audio."
        assert audio.ndim == 1, "Expected mono audio."

        # 确认处理后音频与带噪音频长度一致
        noisy_path = audio_dir / f"{utterance_id}_noisy.wav"
        assert len(audio) == sf.info(noisy_path).frames, \
            "Audio length mismatch."

        with torch.inference_mode():
            result = model.transcribe(audio, **decode_settings)

        reference = row["reference_text"]
        hypothesis = result["text"].strip()

        condition_results.append({
            "utterance_id": utterance_id,
            "condition": condition,
            "reference_text": reference,
            "hypothesis_text": hypothesis,
            "reference_normalized": normalizer(reference),
            "hypothesis_normalized": normalizer(hypothesis),
        })

        print(f"[{index}/10] {utterance_id}")
        print("Whisper:", hypothesis)

    # 计算该条件的整体 WER
    condition_scores = jiwer.process_words(
        [row["reference_normalized"] for row in condition_results],
        [row["hypothesis_normalized"] for row in condition_results],
    )
    classical_scores[condition] = condition_scores

    # 保存识别文本
    output_path = output_dir / f"{condition}_transcripts.csv"
    with output_path.open(
        "w", encoding="utf-8-sig", newline=""
    ) as file:
        writer = csv.DictWriter(
            file, fieldnames=list(condition_results[0])
        )
        writer.writeheader()
        writer.writerows(condition_results)

    print()
    print(f"{condition} WER: {condition_scores.wer * 100:.2f}%")
    print("Substitutions:", condition_scores.substitutions)
    print("Deletions:", condition_scores.deletions)
    print("Insertions:", condition_scores.insertions)
    print()

# 汇总并保存目前完成的四组结果
summary_rows = []

for condition, scores in [
    ("clean", clean_scores),
    ("noisy", noisy_scores),
    ("spectral_subtraction", classical_scores["spectral_subtraction"]),
    ("wiener", classical_scores["wiener"]),
]:
    summary_rows.append({
        "condition": condition,
        "wer_percent": scores.wer * 100,
        "substitutions": scores.substitutions,
        "deletions": scores.deletions,
        "insertions": scores.insertions,
        "reference_words": scores.hits
            + scores.substitutions + scores.deletions,
    })
    print(f"{condition}: WER = {scores.wer * 100:.2f}%")

with (output_dir / "wer_summary_partial.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

print("Four-condition results saved.")

Processing condition: spectral_subtraction
[1/10] 1272-135031-0000
Whisper: Because you are sleeping instead of conquering, the lovely rose princess has become a fiddle without a bow, while poor Shaggy sits there accuing dove.
[2/10] 1462-170142-0000
Whisper: The last two days of the voyage, Bartley found almost intolerable.
[3/10] 174-168635-0000
Whisper: He had never been father, lover, husband, friend.
[4/10] 1988-147956-0000
Whisper: Folks brought up a sack of potatoes and a piece of cured pork from the cellar, and grandmother packed some loaves of Saturday's bread, a jar of butter, and several pumpkin pies in a straw of the wagon box.
[5/10] 1993-147964-0000
Whisper: They sought about the house most of the day, as if it were Sunday, greasing their boots, mending their suspenders, plating whiplashes.
[6/10] 2035-147960-0000
Whisper: She was four years older than I, to be sure, and had seen more of the world. But I was a boy, and she was a girl, and I resented her protecting manner.

In [9]:
# 下载官方降噪程序和预训练模型
import urllib.request
import subprocess
import hashlib
from pathlib import Path

tool_dir = Path("/content/deepfilter_tools")
tool_dir.mkdir(parents=True, exist_ok=True)

binary_path = tool_dir / "deep-filter"
model_path = tool_dir / "DeepFilterNet2_onnx.tar.gz"

binary_url = (
    "https://github.com/Rikorose/DeepFilterNet/releases/"
    "download/v0.5.6/"
    "deep-filter-0.5.6-x86_64-unknown-linux-musl"
)

model_url = (
    "https://raw.githubusercontent.com/Rikorose/DeepFilterNet/"
    "v0.5.6/models/DeepFilterNet2_onnx.tar.gz"
)

for url, path in [
    (binary_url, binary_path),
    (model_url, model_path),
]:
    if not path.exists():
        print("Downloading:", path.name)
        urllib.request.urlretrieve(url, path)

binary_path.chmod(0o755)

# 检查程序是否能正常运行
result = subprocess.run(
    [str(binary_path), "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout.strip())

# 保存文件校验值，方便记录所用版本
metadata = {
    "release": "v0.5.6",
    "model": "DeepFilterNet2",
    "binary_url": binary_url,
    "model_url": model_url,
    "binary_sha256": hashlib.sha256(
        binary_path.read_bytes()
    ).hexdigest(),
    "model_sha256": hashlib.sha256(
        model_path.read_bytes()
    ).hexdigest(),
}

with (output_dir / "deepfilter_settings.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2)

print("DeepFilterNet setup completed.")

Downloading: deep-filter
Downloading: DeepFilterNet2_onnx.tar.gz
deep_filter 0.5.6
DeepFilterNet setup completed.


In [11]:
# 使用官方程序自带的 DeepFilterNet3 处理 10 条带噪语音
import csv
import json
import subprocess
import hashlib
from pathlib import Path

import numpy as np
import scipy
import soundfile as sf
from scipy.signal import resample_poly

# 设置目录
pilot_dir = Path("/content/ELEC5305_Pilot")
audio_dir = pilot_dir / "results" / "audio"
output_dir = pilot_dir / "results" / "asr"
processing_dir = pilot_dir / "results" / "deepfilter_processing"
binary_path = Path("/content/deepfilter_tools/deep-filter")

output_dir.mkdir(parents=True, exist_ok=True)
processing_dir.mkdir(parents=True, exist_ok=True)

assert binary_path.is_file(), "DeepFilterNet executable not found."

# 读取 10 条音频的清单
with (pilot_dir / "utterances.csv").open(
    encoding="utf-8-sig", newline=""
) as file:
    manifest = list(csv.DictReader(file))

assert len(manifest) == 10, "Expected 10 utterances."

deepfilter_checks = []

for index, row in enumerate(manifest, start=1):
    utterance_id = row["utterance_id"]
    noisy_path = audio_dir / f"{utterance_id}_noisy.wav"

    noisy_audio, sample_rate = sf.read(
        noisy_path, dtype="float32"
    )
    assert sample_rate == 16000, "Expected 16 kHz audio."
    assert noisy_audio.ndim == 1, "Expected mono audio."
    assert np.isfinite(noisy_audio).all(), "Invalid input samples."

    # 从 16 kHz 转为 48 kHz，不调整音量
    audio_48k = resample_poly(noisy_audio, 3, 1)
    original_length_48k = len(audio_48k)

    # 末尾补一秒零，避免延迟补偿后丢失句子结尾
    padded_audio = np.concatenate([
        audio_48k,
        np.zeros(48000, dtype="float32"),
    ])

    input_path = processing_dir / f"{utterance_id}_48k.wav"
    sf.write(input_path, padded_audio, 48000, subtype="FLOAT")

    enhanced_dir = processing_dir / utterance_id
    enhanced_dir.mkdir(parents=True, exist_ok=True)
    enhanced_path = enhanced_dir / input_path.name

    # 删除上次可能留下的中间输出
    if enhanced_path.exists():
        enhanced_path.unlink()

    # 不指定外部模型，使用程序自带的 DeepFilterNet3
    # 每条音频单独运行，模型状态不会沿用到下一条
    command = [
        str(binary_path),
        "-D",
        "-o", str(enhanced_dir),
        str(input_path),
    ]

    process = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if process.returncode != 0:
        print(process.stdout)
        print(process.stderr)
        raise RuntimeError(f"Enhancement failed: {utterance_id}")

    assert enhanced_path.is_file(), "Enhanced output not found."

    enhanced_48k, enhanced_rate = sf.read(
        enhanced_path, dtype="float32"
    )

    assert enhanced_rate == 48000, "Unexpected sample rate."
    assert enhanced_48k.ndim == 1, "Expected mono output."
    assert len(enhanced_48k) >= original_length_48k, \
        "Enhanced audio is too short."

    # -D 已补偿延迟，这里仅移除末尾补零部分
    enhanced_48k = enhanced_48k[:original_length_48k]

    # 转回 16 kHz，检查长度和数值
    enhanced_16k = resample_poly(enhanced_48k, 1, 3)

    assert len(enhanced_16k) == len(noisy_audio), \
        "Audio length mismatch."
    assert np.isfinite(enhanced_16k).all(), \
        "Invalid output samples."

    # 使用浮点 WAV 保存，不单独归一化或截断峰值
    output_path = audio_dir / f"{utterance_id}_deepfilter.wav"
    sf.write(output_path, enhanced_16k, 16000, subtype="FLOAT")

    deepfilter_checks.append({
        "utterance_id": utterance_id,
        "input_samples": len(noisy_audio),
        "output_samples": len(enhanced_16k),
        "output_peak": float(np.max(np.abs(enhanced_16k))),
    })

    print(f"[{index}/10] {utterance_id}: enhancement completed")

# 保存音频检查结果
with (output_dir / "deepfilter_audio_checks.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(
        file, fieldnames=list(deepfilter_checks[0])
    )
    writer.writeheader()
    writer.writerows(deepfilter_checks)

# 保存实际使用的模型和处理设置
metadata = {
    "release": "v0.5.6",
    "model": "DeepFilterNet3",
    "model_source": "Embedded in the official v0.5.6 binary",
    "binary_sha256": hashlib.sha256(
        binary_path.read_bytes()
    ).hexdigest(),
    "resampling": "scipy.signal.resample_poly",
    "scipy_version": scipy.__version__,
    "sample_rates": "16000 -> 48000 -> 16000",
    "delay_compensation": True,
    "tail_padding_seconds": 1,
    "post_filter": False,
    "output_format": "FLOAT WAV",
}

with (output_dir / "deepfilter_settings.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2)

# 检查是否已生成全部 10 条降噪音频
for row in manifest:
    output_path = audio_dir / f"{row['utterance_id']}_deepfilter.wav"
    assert output_path.is_file(), "An enhanced file is missing."

print()
print("Model: DeepFilterNet3")
print("Enhanced audio files: 10")
print("DeepFilterNet enhancement completed for 10 utterances.")

[1/10] 1272-135031-0000: enhancement completed
[2/10] 1462-170142-0000: enhancement completed
[3/10] 174-168635-0000: enhancement completed
[4/10] 1988-147956-0000: enhancement completed
[5/10] 1993-147964-0000: enhancement completed
[6/10] 2035-147960-0000: enhancement completed
[7/10] 2412-153948-0000: enhancement completed
[8/10] 2428-83699-0000: enhancement completed
[9/10] 251-118436-0000: enhancement completed
[10/10] 2803-154320-0000: enhancement completed

Model: DeepFilterNet3
Enhanced audio files: 10
DeepFilterNet enhancement completed for 10 utterances.


In [12]:
# 识别 DeepFilterNet3 降噪后的语音
deepfilter_results = []

for index, row in enumerate(manifest, start=1):
    utterance_id = row["utterance_id"]
    audio_path = audio_dir / f"{utterance_id}_deepfilter.wav"

    audio, sample_rate = sf.read(audio_path, dtype="float32")
    assert sample_rate == 16000, "Expected 16 kHz audio."
    assert audio.ndim == 1, "Expected mono audio."
    assert np.isfinite(audio).all(), "Invalid audio samples."

    noisy_path = audio_dir / f"{utterance_id}_noisy.wav"
    assert len(audio) == sf.info(noisy_path).frames, \
        "Audio length mismatch."

    with torch.inference_mode():
        result = model.transcribe(audio, **decode_settings)

    reference = row["reference_text"]
    hypothesis = result["text"].strip()

    deepfilter_results.append({
        "utterance_id": utterance_id,
        "condition": "deepfilternet3",
        "reference_text": reference,
        "hypothesis_text": hypothesis,
        "reference_normalized": normalizer(reference),
        "hypothesis_normalized": normalizer(hypothesis),
    })

    print(f"[{index}/10] {utterance_id}")
    print("Whisper:", hypothesis)

# 保存识别文本
with (output_dir / "deepfilter_transcripts.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(
        file, fieldnames=list(deepfilter_results[0])
    )
    writer.writeheader()
    writer.writerows(deepfilter_results)

# 从五组已保存的文本计算最终结果，不重复识别
transcript_files = [
    ("clean", "clean_transcripts.csv"),
    ("noisy", "noisy_transcripts.csv"),
    ("spectral_subtraction", "spectral_subtraction_transcripts.csv"),
    ("wiener", "wiener_transcripts.csv"),
    ("deepfilternet3", "deepfilter_transcripts.csv"),
]

all_scores = {}
reference_baseline = None

for condition, filename in transcript_files:
    with (output_dir / filename).open(
        encoding="utf-8-sig", newline=""
    ) as file:
        rows = list(csv.DictReader(file))

    assert len(rows) == 10, "Expected 10 transcripts."
    assert [row["utterance_id"] for row in rows] == [
        row["utterance_id"] for row in manifest
    ], "Utterance order mismatch."

    references = [row["reference_normalized"] for row in rows]
    hypotheses = [row["hypothesis_normalized"] for row in rows]

    assert all(references), "An empty reference was found."

    if reference_baseline is None:
        reference_baseline = references
    else:
        assert references == reference_baseline, \
            "Reference transcripts differ between conditions."

    all_scores[condition] = jiwer.process_words(
        references, hypotheses
    )

# 汇总整体 WER 和相对带噪语音的变化
summary_rows = []
noisy_wer = all_scores["noisy"].wer

for condition, scores in all_scores.items():
    summary_rows.append({
        "condition": condition,
        "wer_percent": scores.wer * 100,
        "substitutions": scores.substitutions,
        "deletions": scores.deletions,
        "insertions": scores.insertions,
        "reference_words": (
            scores.hits + scores.substitutions + scores.deletions
        ),
        "delta_wer_pp": (scores.wer - noisy_wer) * 100,
    })

with (output_dir / "wer_summary.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(
        file, fieldnames=list(summary_rows[0])
    )
    writer.writeheader()
    writer.writerows(summary_rows)

# 显示最终五行表
print()
print("Final pilot results")
print(f"{'Condition':<24} {'WER (%)':>9} {'S':>5} {'D':>5} {'I':>5}")

for row in summary_rows:
    print(
        f"{row['condition']:<24} "
        f"{row['wer_percent']:>9.2f} "
        f"{row['substitutions']:>5} "
        f"{row['deletions']:>5} "
        f"{row['insertions']:>5}"
    )

print()
print("Final WER table saved: wer_summary.csv")

[1/10] 1272-135031-0000
Whisper: Because you were sleeping instead of conquering, the lovely rose princess has become a fiddle without a bow, while poor Shaggy sits there, a cooing dove.
[2/10] 1462-170142-0000
Whisper: The last two days of the voyage, Bartley found almost intolerable. You
[3/10] 174-168635-0000
Whisper: He had never been father, lover, husband, friend.
[4/10] 1988-147956-0000
Whisper: Books brought up a sack of potatoes and a piece of cured pork from the cellar, and grandmother packed some loaves of Saturday's bread, a jar of butter, and several pumpkin pies in the straw of the wagon box.
[5/10] 1993-147964-0000
Whisper: They sought about the house most of the day, as if it were Sunday, greasing their boots, their suspenders, plating whiplashes.
[6/10] 2035-147960-0000
Whisper: She was four years older than I, to be sure, and had seen more of the world. But I was a boy, and she was a girl, and I resented her protecting manner.
[7/10] 2412-153948-0000
Whisper: If the r